# AIR Homework 1 (Task 3)

## Setup

This notebook covers Task 3. Run the cells below before you start. They set up the notebook, load our working environment from `scene.py`, import your code, and recreate what we need from Tasks 1 and 2.

**NOTE:** The setup uses your code in `task1.py` and `task2.py`, so complete Tasks 1 and 2 before you start this task.

In [ ]:
%load_ext autoreload
%autoreload 2

# set plot theme
import seaborn as sns
sns.set_theme()

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

In [ ]:
# load our working environment from scene.py (see start_here.ipynb)
from scene import HOME_CONFIG, data, model

import mujoco
import mediapy as media
import numpy as np
from ipywidgets import interact

In [ ]:
# import your code. the setup below uses your code from Tasks 1 and 2 (t1, t2), and
# the rest of this notebook uses your code for this task (t3), e.g. t3.set_state
import task1 as t1
import task2 as t2
import task3 as t3

In [ ]:
# from Task 1
import unittest

# used by the sanity checks
test = unittest.TestCase()


def get_check_data():
    """Get a separate simulation state at the home configuration, so that the sanity
    checks don't affect the rest of this notebook."""
    check_data = mujoco.MjData(model)
    mujoco.mj_resetDataKeyframe(model, check_data, model.key("robot_home").id)
    mujoco.mj_forward(model, check_data)
    return check_data


# list camera options
CAMERA_LIST = [
    "leftsideview",
    "rightsideview",
    "frontview",
    "birdview",
    "leftangleview",
    "rightangleview",
]

# get the qpos indices corresponding to the robot joints
robot_qpos_idx = t1.get_entity_qpos_idx(model, "robot")

# the pre-grasp pose above the red box, at the home keyframe
mujoco.mj_resetDataKeyframe(model, data, model.key("robot_home").id)
mujoco.mj_forward(model, data)
above_block_pos, above_block_rot_mat = t1.get_above_block_pose(data)

## Task 3: Motion Planning for Manipulation

In motion planning tutorial, we used the Open Motion Planning Library (OMPL) to bring a robot from a start configuration to a goal configuration while avoiding obstacles. However, this only included static environments where the robot's only task was to navigate from point A to point B (in C-space). In manipulation tasks, the robot explicit goal is to interact with objects, thus changing the environment.

We can view our manipulation task as a sequence of motion planning problems. This problem is often referred to as **task and motion planning (TAMP)**, where the higher-level task is treated as a separate discrete planning problem. We will cover TAMP in more detail later in the course. For now, let's manually break down our pick-and-place task:
1. Move from initial configuration to home configuration
2. Move from home configuration to pre-grasp configuration (e.g., above the red box)
3. Move from pre-grasp configuration to grasp configuration (to grab hold of the red box)
4. Move from grasp configuration to pre-grasp configuration (to lift the red box)
5. Move from pre-grasp configuration to place configuration (e.g., above the green bin)
6. Move from place configuration to home configuration

Let's implement the components needed to solve each motion planning problem in the sequence above, step by step.

### Collision Checking in Changing Environments

One thing all the above motion planning problems have in common is the need for collision checking. In the previous motion planning tutorial, we implemented a simple collision checker that checked for collisions between the robot and static obstacles in the environment. However, in our pick-and-place task, we have movable obstacles (the red box and green bin) that change position as the robot interacts with them.

To account for the changing environment, we need a parameterized collision checker that changes which obstacles to check for collisions based on the current step in the manipulation sequence. We differentiate between three kinds of bodies in the environment:
1. **Controlled Bodies**: the object for which we are planning the motion. They may not collide.
2. **Obstacles**: all other objects in the environment with which the controlled bodies must avoid colliding.
3. **Attached bodies**: bodies that are allowed to be in contact (e.g., the gripper and the red box when grasping).

Let's implement a parameterized collision checker that takes in the moving body and colliding bodies as parameters:

**TODO:** implement `is_collision_state` in `task3.py`.

In [ ]:
# sanity check: is_collision_state
check_data = get_check_data()
robot_and_gripper = ["robot_", "gripper_"]
table_collision_config = [-3.0, 0.3, 0, -1.5708, -1.5708, 0]  # the forearm is inside the table

# there are no collisions at the home configuration
test.assertFalse(t3.is_collision_state(model, check_data, robot_and_gripper))

# there is a collision when the forearm is inside the table
check_data.qpos[t1.get_entity_qpos_idx(model, "robot")] = table_collision_config
mujoco.mj_forward(model, check_data)
test.assertTrue(t3.is_collision_state(model, check_data, robot_and_gripper))
test.assertTrue(t3.is_collision_state(model, check_data, ["table"]))

# it doesn't count if we exclude it (in any order) or if it doesn't involve the controlled bodies
test.assertFalse(t3.is_collision_state(model, check_data, robot_and_gripper, [("robot_", "table")]))
test.assertFalse(t3.is_collision_state(model, check_data, robot_and_gripper, [("table", "robot_")]))
test.assertFalse(t3.is_collision_state(model, check_data, ["gripper_"]))

print("is_collision_state sanity checks passed. forearm in the table detected, home is free")

In the motion planning tutorial, to check if a state is valid, we would first set the robot configuration to the given state and then check for collisions. In a manipulation task, we must consider that the robot may be holding an object which also needs to be updated in the simulation. However, changing only the robot `qpos` value will leave the held object's pose unchanged. Therefore, we need to update the held object's pose based on the robot's end-effector pose after setting the robot configuration.

Let's implement a function to set the state of the simulation given the robot configuration and the held object's name (if any).

**Hint**: This is another instance of reference frame transformations!

**TODO:** implement `set_state` in `task3.py`.

In [ ]:
# sanity check: set_state
robot_idx = t1.get_entity_qpos_idx(model, "robot")
turned_config = HOME_CONFIG + [0.5, 0, 0, 0, 0, 0]  # turn the robot around its base


def get_box_pose_from_ee(check_data):
    ee_pos, ee_rot_mat = t1.get_ee_pose(check_data)
    box = check_data.body("red_box")
    return ee_rot_mat.T @ (box.xpos - ee_pos), ee_rot_mat.T @ box.xmat.reshape(3, 3)


# without an attached body, the robot moves and the box stays where it is
check_data = get_check_data()
box_pos = check_data.body("red_box").xpos.copy()
t3.set_state(model, check_data, turned_config, robot_idx)
np.testing.assert_allclose(check_data.qpos[robot_idx], turned_config)
np.testing.assert_allclose(check_data.body("red_box").xpos, box_pos)

# with an attached body, the box keeps its pose relative to the end-effector
check_data = get_check_data()
start_pos, start_rot_mat = get_box_pose_from_ee(check_data)
t3.set_state(model, check_data, turned_config, robot_idx, attached_body_name="red_box")
end_pos, end_rot_mat = get_box_pose_from_ee(check_data)
np.testing.assert_allclose(end_pos, start_pos, atol=1e-6)
np.testing.assert_allclose(end_rot_mat, start_rot_mat, atol=1e-6)

print(f"set_state sanity checks passed. box position relative to the end-effector = {np.round(end_pos, 3)}")

### Grasping Configurations

Why do we require a pre-grasp AND a grasp configuration? The pre-grasp configuration allows the robot to approach the object from a safe distance, reducing the risk of collisions with the object or surrounding obstacles. It also provides a controlled approach path, which is essential for precise manipulation tasks. The grasp configuration is where the robot actually makes contact with the object to be manipulated. By separating these two configurations, we can ensure that the robot can position itself accurately before executing the grasp, leading to more reliable and successful manipulation.

We already found one pre-grasp configuration above the red box using IK, saved in the `above_block_pos` and `above_block_rot_mat` variables. However, there is another configuration above the red box that we can use:

**TODO:** implement `get_above_block_rot_mat2` in `task3.py`.

In [ ]:
mujoco.mj_resetData(model, data)  # Reset state and time.

# update the Mujoco data structure to reflect the reset state
# before calling the IK for the first time.
mujoco.mj_forward(model, data)

# set an alternative orientation above the box (90 degrees rotated around z axis)
above_block_rot_mat2 = t3.get_above_block_rot_mat2(above_block_rot_mat)

# compute IK solutions for the above-box end-effector pose
box_ik_solutions2 = t1.compute_ik(data, above_block_pos, above_block_rot_mat2)


# interactive visualization of IK solutions for above-box pose
@interact(ik_solution_number=(0, len(box_ik_solutions2) - 1), camera=CAMERA_LIST)
def render_ik_solution(ik_solution_number, camera="leftsideview"):
    # apply one of the IK solutions
    data.qpos[robot_qpos_idx] = box_ik_solutions2[ik_solution_number]
    mujoco.mj_forward(model, data)  # Update the simulation state
    with mujoco.Renderer(model) as renderer:
        renderer.update_scene(data, camera=camera)
        media.show_image(renderer.render())
        if not t3.is_collision_state(
            model, data, controlled_bodies_prefix=["robot_", "gripper_"], verbose=True
        ):
            print("No collision detected")

There are four more possible pre-grasp configurations, one for each side of the box. In the general case, we would like to evaluate all possible grasping configurations to find the best one. For simplicity, we will only consider the two configurations above the box for this assignment.

We now need to select a grasp configuration. There are many considerations in this selection process, including:
1. **Stability**: will the robot be able to hold onto the object securely?
2. **Reachability**: can the robot reach the grasp configuration without colliding with obstacles?
3. **Task Requirements**: does the grasp configuration allow the robot to perform the subsequent manipulation tasks effectively?

The above grasp quality measures are typically combined into a single "ranking pipeline". However, for simplicity, we will use a simple heuristic: Choose the closest configuration to the robot's current configuration (Euclidean) that is not in collision with any obstacles. Here, we make the assumption that Euclidean distance in configuration space is a reasonable proxy for the actual effort required to move between configurations, and that the target configuration is reachable.

Let's implement this function and select a pre-grasp configuration. Remember, We can compute IK efficiently using our analytical solver, and collision detection is relatively expensive.

**TODO:** implement `select_pose_configuration` in `task3.py`.

In [ ]:
# define pre-grasp poses
pre_grasp_poses = [
    (above_block_pos, above_block_rot_mat),
    (above_block_pos, above_block_rot_mat2),
]

# select a collision-free pre-grasp configuration
mujoco.mj_resetData(model, data)

# update the Mujoco data structure to reflect the reset state
# before calling the IK for the first time.
mujoco.mj_forward(model, data)

# select collision-free pre-grasp configuration
selected_pre_grasp_config = t3.select_pose_configuration(
    model,
    data,
    pre_grasp_poses,
    controlled_bodies_prefix=["robot_", "gripper_"],
    exclude_collision_prefix_pairs=[("gripper_", "red_box")],
)
if selected_pre_grasp_config is not None:
    print("Selected pre-grasp configuration:", selected_pre_grasp_config)
    data.qpos[t1.get_entity_qpos_idx(model, "robot")] = selected_pre_grasp_config
    mujoco.mj_forward(model, data)
    with mujoco.Renderer(model) as renderer:
        renderer.update_scene(data, camera="frontview")
        media.show_image(renderer.render())
else:
    print("No collision-free pre-grasp configuration found.")

Using the same logic, let's find a grasp configuration. This corresponds to poses identical to the pre-grasp configurations, but lowered down to the top of the red box.

**TODO:** implement `get_on_block_pos` in `task3.py`.

In [ ]:
# define grasp poses to be slightly lower than pre-grasp but not touching the box
on_block_pos = t3.get_on_block_pos(above_block_pos)
grasp_poses = [
    (on_block_pos, above_block_rot_mat),
    (on_block_pos, above_block_rot_mat2),
]

# select a collision-free grasp configuration
selected_grasp_config = t3.select_pose_configuration(
    model, data, grasp_poses, controlled_bodies_prefix=["robot_", "gripper_"]
)
if selected_grasp_config is not None:
    print("Selected grasp configuration:", selected_grasp_config)
    data.qpos[t1.get_entity_qpos_idx(model, "robot")] = selected_grasp_config
    mujoco.mj_forward(model, data)
    with mujoco.Renderer(model) as renderer:
        renderer.update_scene(data, camera="frontview")
        media.show_image(renderer.render())
else:
    print("No collision-free grasp configuration found.")

To lower the end-effector from the pre-grasp configuration to the grasp configuration, we will take a leap of faith and assume that a straight-line motion in C-space is collision-free. This way, we can use our PID controller to move the end-effector between the two configurations. However, this is not always the case, requiring additional motion planning. We will ignore this for now.

In [ ]:
# set the robot to the selected pre-grasp configuration
mujoco.mj_resetData(model, data)
data.qpos[t1.get_entity_qpos_idx(model, "robot")] = selected_pre_grasp_config
mujoco.mj_forward(model, data)

# create a PID control policy to move to the selected grasp configuration
go_to_grasp_policy = t2.PIDJointPositionControlPolicy(
    model, target_qpos=selected_grasp_config, kp=t2.KP, ki=t2.KI, kd=t2.KD
)

# render the execution of the policy starting from the pre-grasp configuration
# and moving to the grasp configuration.

duration= 1.0 # seconds
framerate = 60  # frames per second

frames = []
with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        go_to_grasp_policy.set_control(data)

        mujoco.mj_step(model, data)

        if len(frames) < data.time * framerate:
            renderer.update_scene(data, camera="frontview")
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

### Gripper Usage

The gripper attached to the robot is a [Robotiq 2F-85](https://www.universal-robots.com/marketplace/products/01tP40000071NgHIAU/) adaptive electric gripper.
- "2F" - refers to the "two-finger" design of the gripper.
- "85" - indicates the maximum opening width of the gripper in millimeters (85 mm).
To control the gripper, we can set the position of the two fingers using the `ctrl` values in the simulation.

While the gripper has many joints that allow for adaptive gripping, it is controlled using a single control signal that sets the opening width of the gripper. This control signal is an integer that ranges from 0 (fully open) to 255 (fully closed). The reason for this is mostly for efficiency: 0-255 can be represented using a single byte, allowing for faster communication between the robot and the controller.

For the purpose of this assignment, we will only use two gripper positions: fully open and fully closed. Let's implement a controller for the gripper:

**TODO:** implement `GripperController` in `task3.py`.

We need to ensure that the robot is stationary when operating the gripper. Let's implement a policy that does this:

`StayInPlacePolicy` is provided in `task3.py`.

Let's write a short script to test out our gripper controller:

In [ ]:
# set the robot to begin in the grasp configuration
mujoco.mj_resetData(model, data)
data.qpos[t1.get_entity_qpos_idx(model, "robot")] = selected_grasp_config
mujoco.mj_forward(model, data)

# create the controllers required for box lifting
# 1. close the gripper on the box
# 2. stay in place while the gripper is closing
# 3. go to the pre-grasp configuration while grasping the box to lift
gripper_controller = t3.GripperController(model)
stay_in_place_policy = t3.StayInPlacePolicy(model)
go_to_pregrasp_policy = t2.PIDJointPositionControlPolicy(
    model, target_qpos=selected_pre_grasp_config,
    kp=t2.KP, ki=t2.KI, kd=t2.KD
)

# set the duration for gripping and lifting
# together they define the total simulation duration
grip_duration= 1.0 # seconds
lift_duration = 2.0  # seconds

framerate = 60  # frames per second

frames = []
with mujoco.Renderer(model) as renderer:
    # simulate until allocated durations for gripping and lifting are done
    while data.time < grip_duration + lift_duration:

        # start with gripping phase. stay in place while closing gripper
        if data.time < grip_duration:
            stay_in_place_policy.set_control(data)
            gripper_controller.close_gripper(data)

        # after gripping, go to pre-grasp to lift the box
        else:
            go_to_pregrasp_policy.set_control(data)

        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:
            renderer.update_scene(data, camera="leftsideview")
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

### Motion Planner Setup

Let's implement the motion planning pipeline.

We start by defining the state space for the motion planner. This includes the robot's configuration limits. In the motion planning tutorial, we simply accepted the limits as is. However, we can apply a simple trick to reduce the size of the state space and improve planning times. Note that the UR5e robot has joint limits that allow for multiple full rotations of some joints. While this can be useful in practice, we can achieve any desired end-effector pose using a single rotation of each joint, which is enough for our task. Therefore, we can tighten the joint limits to reduce the size of the state space without losing any reachable end-effector poses.

**TODO:** implement `get_clipped_joint_bounds` in `task3.py`.

In [ ]:
t3.get_clipped_joint_bounds(model)

In [ ]:
# sanity check: get_clipped_joint_bounds
joint_bounds = t3.get_clipped_joint_bounds(model)

# one range for each robot joint, none of them longer than a single rotation
test.assertEqual(len(joint_bounds), 6)
for low, high in joint_bounds:
    test.assertLess(low, high)
    test.assertGreaterEqual(low, -np.pi)
    test.assertLessEqual(high, np.pi)

print(
    "get_clipped_joint_bounds sanity checks passed. "
    f"widest range = {max(high - low for low, high in joint_bounds):.3f} radians"
)

Let's define the state space with tightened joint limits:

`get_ur5e_state_space` is provided in `task3.py`.

Remember that an important part of the state space is the free space. Let's implement a `StateValidityChecker` as we did in the motion planning tutorial, but using our parameterized collision checker and dynamic state setting function that considers attached bodies.

**TODO:** implement `UR5eValidityChecker` in `task3.py`.

In [ ]:
# sanity check: UR5eValidityChecker
from ompl import base as ob

# OMPL states for the home configuration and the configuration with the forearm in the table
state_space = t3.get_ur5e_state_space(model)
home_state = ob.State(state_space)
collision_state = ob.State(state_space)
for i in range(6):
    home_state[i] = HOME_CONFIG[i]
    collision_state[i] = table_collision_config[i]

state_info = ob.SpaceInformation(state_space)
validity_checker = t3.UR5eValidityChecker(state_info, model, get_check_data(), robot_and_gripper)
test.assertTrue(validity_checker.isValid(home_state()))
test.assertFalse(validity_checker.isValid(collision_state()))

print("UR5eValidityChecker sanity checks passed. home is valid, forearm in the table is invalid")

Now let's implement a function that gives us the state space info object with the desired validity checker and checking resolution:

**TODO:** implement `get_state_space_info` in `task3.py`.

In [ ]:
# sanity check: get_state_space_info
state_info = t3.get_state_space_info(model, get_check_data(), robot_and_gripper, check_resolution=0.02)
test.assertAlmostEqual(state_info.getStateValidityCheckingResolution(), 0.02)
test.assertTrue(state_info.isValid(home_state()))
test.assertFalse(state_info.isValid(collision_state()))

# the forearm in the table is valid when we exclude this collision
state_info = t3.get_state_space_info(model, get_check_data(), robot_and_gripper, [("robot_", "table")])
test.assertTrue(state_info.isValid(collision_state()))

print("get_state_space_info sanity checks passed. resolution and excluded collisions are in use")

Now, initialize the problem definition as in the motion planning tutorial:

`get_mp_problem` is provided in `task3.py`.

We will now initialize the planner. You can choose any planner in the OMPL library from the list of [available planners](https://ompl.kavrakilab.org/planners.html). Once you have selected a planner, set its parameters as needed. You can see the available parameters for each planner by checking it's documentation. For example, RRTstar parameters are documented [here](https://ompl.kavrakilab.org/classompl_1_1geometric_1_1RRTstar.html#gRRTstar) (you can reach this by clicking the link in the available planners list).

**TODO:** implement `get_planner_for_problem` in `task3.py`.

Let's implement the function that runs the planner and returns a list of waypoints if a solution is found. This is the same as in the motion planning tutorial, except we will also return the solution status. This is an important indicator of whether the planner found a solution and of what quality. The status is represented as an enum. It has many values, but here are some of the most important ones to know about:
- `ompl.base.PlannerStatus.EXACT_SOLUTION` - An exact solution was found
- `ompl.base.PlannerStatus.APPROXIMATE_SOLUTION` - An approximate solution was found

Any other status indicates that no solution was found. Let's implement the function.

`plan_motion` is provided in `task3.py`.

Let's test out our motion planner by planning a collision-free path from the start configuration to the home configuration:

In [ ]:
mujoco.mj_resetData(model, data)  # Reset state and time.

# update the Mujoco data structure to reflect the reset state
# before calling the planner for the first time.
mujoco.mj_forward(model, data)

# create the planner for moving from the initial configuration to the
# home configuration
planner = t3.get_planner_for_problem(
    start_q=[0, 0, 0, 0, 0, 0],
    goal_q=HOME_CONFIG,
    model=model,
    data=data,
    controlled_bodies_prefix=["robot_", "gripper_"],
    check_resolution=0.01,
)

# plan the motion with a time limit of 5 seconds
waypoints, status = t3.plan_motion(planner, time_limit=5.0)

Let's visualize the waypoints returned by the planner:

In [ ]:
mujoco.mj_resetData(model, data)

# interactive visualization of the motion plan waypoints
@interact(wp_id=(0, len(waypoints)-1), camera=CAMERA_LIST)
def render_motion_plan(wp_id, camera="leftsideview"):
    # apply one of the motion plan waypoints
    data.qpos[t1.get_entity_qpos_idx(model, "robot") ] = waypoints[wp_id]
    mujoco.mj_forward(model, data)  # Update the simulation state
    with mujoco.Renderer(model) as renderer:
        renderer.update_scene(data, camera=camera)
        media.show_image(renderer.render())
        print('collision detected:', t3.is_collision_state(
            model, data,
            controlled_bodies_prefix=["robot_", "gripper_"],
        ))

The above code is a great way to debug the planner. You can visualize the waypoints and check for collisions along the path. However, this does not guarantee that the robot will not collide with any obstacles during execution, since this heavily relies on the resolution of the collision checking during planning and the accuracy of the controller.

**Next:** continue to Task 4 in `task4.ipynb`.